**Task 1.1**

In [1]:
# Import the libraries
import csv
import json

In [2]:
# Read the CSV file
with open("/content/appointments.csv", "r", newline="") as file:

    reader = csv.DictReader(file)

    rows = list(reader)

    print("Column headers:")
    print(reader.fieldnames)

    print("\nTotal number of rows:", len(rows))

Column headers:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

Total number of rows: 1020


In [3]:
# Create the parse_row() function

def parse_row(row):

    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

In [4]:
# Separate accepted and rejected records

accepted = []
rejected = []

for row in rows:

    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

Accepted rows: 988
Rejected rows: 32


In [5]:
# Save rejected records

with open("rejected_rows.json", "w") as file:

    json.dump(rejected, file, indent=2)

print("Rejected records saved successfully!")

Rejected records saved successfully!


**Task 1.2: Initial data cleaning**

In [6]:
# Import pandas and NumPy

import pandas as pd
import numpy as np

In [7]:
# Load the original dataset

df = pd.read_csv("/content/appointments.csv")

print("Dataset shape:", df.shape)

Dataset shape: (1020, 14)


In [8]:
# Inspect the dataset

print("Dataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patient_id          0
clinic_id 

In [9]:
# Remove exact duplicate rows

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1000


In [10]:
# Standardize booking_channel

df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print(df["booking_channel"].value_counts())
print("\nDistinct channels:",
      df["booking_channel"].nunique())

booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64

Distinct channels: 3


In [11]:
# Convert date columns

df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)

print(df[["booked_on", "appt_date"]].dtypes)

booked_on    datetime64[ns]
appt_date    datetime64[ns]
dtype: object


In [12]:
# Remove records with missing dates or status

df = df.dropna(
    subset=["appt_date", "status"]
)

print("Final rows after Task 1.2:", len(df))

Final rows after Task 1.2: 990


**Task 1.3**

In [13]:
# Identify invalid ages

invalid_age_count = (
    (df["age"] < 1) | (df["age"] > 100)
).sum()

print("Invalid ages:", invalid_age_count)

Invalid ages: 5


In [14]:


df.loc[
    (df["age"] < 1) | (df["age"] > 100),
    "age"
] = np.nan

In [15]:
# Fill missing age and distance

age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("Age median:", age_median)
print("Distance median:", distance_median)

Age median: 48.0
Distance median: 8.25


In [16]:
df["age"] = (
    df["age"]
    .fillna(age_median)
    .round()
    .astype(int)
)

df["distance_km"] = (
    df["distance_km"]
    .fillna(distance_median)
)

In [17]:
# Detect and cap distance outliers

Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

print("Q1:", Q1_distance)
print("Q3:", Q3_distance)
print("IQR:", IQR_distance)
print("Lower fence:", lower_distance)
print("Upper fence:", upper_distance)

Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower fence: -7.9375
Upper fence: 25.5625


In [18]:
distance_capped = (
    (df["distance_km"] < lower_distance) |
    (df["distance_km"] > upper_distance)
).sum()

print("Distance values to cap:", distance_capped)

Distance values to cap: 26


In [19]:
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

In [20]:
# Detect and cap fee outliers

Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

print("Fee Q1:", Q1_fee)
print("Fee Q3:", Q3_fee)
print("Fee IQR:", IQR_fee)
print("Fee lower fence:", lower_fee)
print("Fee upper fence:", upper_fee)

Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Fee lower fence: 68.75
Fee upper fence: 1218.75


In [21]:
fee_capped = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
).sum()

print("Fee values to cap:", fee_capped)

Fee values to cap: 3


In [22]:
df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

In [23]:
# Round the values

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = (
    df["fee_inr"].round().astype(int)
)

print(df[["age", "distance_km", "fee_inr"]].head())

   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700


In [24]:
# Final verification

print("Invalid ages remaining:",
      ((df["age"] < 1) | (df["age"] > 100)).sum())

print("\nMissing values:")
print(df[["age", "distance_km", "fee_inr"]].isnull().sum())

print("\nDistance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

print("\nLargest fee after capping:",
      df["fee_inr"].max())

Invalid ages remaining: 0

Missing values:
age            0
distance_km    0
fee_inr        0
dtype: int64

Distance values capped: 26
Fee values capped: 3

Largest fee after capping: 1219


**Task 1.4 — Feature engineering and NumPy**

In [25]:
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

print(df["lead_days"].head())

0     1
1    10
2     6
4     4
5    40
Name: lead_days, dtype: int64


In [26]:
print("Minimum lead days:", df["lead_days"].min())
print("Maximum lead days:", df["lead_days"].max())

Minimum lead days: 1
Maximum lead days: 45


In [27]:
# Create no_show

df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

print(df[["status", "no_show"]].head(10))

      status  no_show
0    No-Show        1
1   Attended        0
2   Attended        0
4   Attended        0
5    No-Show        1
6   Attended        0
7   Attended        0
8   Attended        0
9   Attended        0
10  Attended        0


In [28]:
print(df["no_show"].value_counts())

no_show
0    772
1    218
Name: count, dtype: int64


In [29]:
# Prepare the NumPy feature matrix

X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

print("Shape of X:", X.shape)
print(X[:5])

Shape of X: (990, 4)
[[ 60.   18.  850.    1. ]
 [ 39.   12.5 350.   10. ]
 [ 63.    4.1 850.    6. ]
 [ 52.    7.5 750.    4. ]
 [ 64.   19.  700.   40. ]]


In [30]:
# Calculate the mean of each column

column_means = np.mean(X, axis=0)

print("Column means:", column_means)

Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [31]:
# Find appointments booked more than 14 days ahead

mask = df["lead_days"] > 14

print("Appointments above 14 days:", mask.sum())

print("Share of appointments:", mask.mean())

Appointments above 14 days: 296
Share of appointments: 0.298989898989899


In [32]:
print("Percentage:",
      round(mask.mean() * 100, 1), "%")

Percentage: 29.9 %


In [33]:
# Calculate distance z-scores
x = df["distance_km"].to_numpy()

z_scores = (
    x - np.mean(x)
) / np.std(x)

print("Z-score count:",
      np.sum(np.abs(z_scores) > 2))

Z-score count: 53


In [34]:
# Final verification

print("Total appointments:", len(df))

print("No-shows:", df["no_show"].sum())

print("Lead days range:",
      df["lead_days"].min(),
      "to",
      df["lead_days"].max())

print("Mean values:", np.mean(X, axis=0))

print("Appointments with lead_days > 14:",
      mask.sum())

print("Percentage above 14 days:",
      round(mask.mean() * 100, 1))

print("Distance z-score outliers:",
      np.sum(np.abs(z_scores) > 2))

Total appointments: 990
No-shows: 218
Lead days range: 1 to 45
Mean values: [ 48.17373737   9.50090909 635.71414141  11.81919192]
Appointments with lead_days > 14: 296
Percentage above 14 days: 29.9
Distance z-score outliers: 53


**Task 1.5 — Live API enrichment using Open-Meteo**

In [35]:
# Import pandas for reading and handling CSV files
import pandas as pd

# Import requests to make HTTP/API requests
import requests

# Read the original clinic data from the CSV file
clinics = pd.read_csv("/content/clinics.csv")

# Display the number of rows and columns
print(clinics.shape)

# Display all column names in the dataset
print(clinics.columns)

# Display the first five rows to inspect the data
display(clinics.head())

(8, 6)
Index(['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude'], dtype='object')


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873


In [36]:
# Select the first clinic from the dataset
clinic = clinics.iloc[0]

# Store the Open-Meteo forecast API URL
url = "https://api.open-meteo.com/v1/forecast"

# Prepare the parameters required by the API
params = {
    "latitude": clinic["latitude"],       # Clinic's latitude
    "longitude": clinic["longitude"],     # Clinic's longitude
    "hourly": "temperature_2m",            # Request hourly temperature at 2 metres
    "forecast_days": 1                     # Request one day's forecast
}

# Send a GET request to the API with a 10-second timeout
response = requests.get(url, params=params, timeout=10)

# Display the HTTP response status code
# 200 means the request was successful
print(response.status_code)

200


In [37]:
# Convert the API response from JSON into a Python dictionary
data = response.json()

# Access the hourly temperature list from the response
temperatures = data["hourly"]["temperature_2m"]

# Extract the first temperature value from the list
temp_c = temperatures[0]

# Display the extracted temperature in Celsius
print("Temperature:", temp_c, "°C")

Temperature: 17.9 °C


In [38]:
# Create an empty list to store the enriched clinic records
enriched_rows = []

# Loop through every clinic in the original dataset
for _, clinic in clinics.iterrows():

    # Initially assume the temperature is unavailable
    temp_c = None

    # Variable to store the reason if the API request fails
    error_reason = None

    # Prepare API parameters using the current clinic's coordinates
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    # Allow a maximum of two attempts:
    # First attempt and one retry
    for attempt in range(2):

        try:
            # Send a GET request to the Open-Meteo API
            # Timeout prevents the program from waiting indefinitely
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            # Check whether the API returned a successful status code
            if response.status_code != 200:

                # Store the HTTP error for reporting
                error_reason = f"HTTP status {response.status_code}"

                # Skip to the next attempt
                continue

            # Convert the successful API response into a dictionary
            data = response.json()

            # Safely access the hourly temperature list
            # Return an empty list if the required data is missing
            temperatures = data.get("hourly", {}).get(
                "temperature_2m", []
            )

            # Check whether at least one temperature value exists
            if len(temperatures) > 0:

                # Extract the first hourly temperature
                temp_c = temperatures[0]

                # Stop retrying because the request succeeded
                break

            else:
                # Record an error if hourly temperature data is missing
                error_reason = "Hourly temperature data missing"

        # Handle network-related errors and request timeouts
        except requests.exceptions.RequestException as e:

            # Store the error message for reporting
            error_reason = str(e)

        # Handle invalid JSON responses
        except ValueError:

            # Store the reason for the failure
            error_reason = "Invalid JSON response"

    # Check whether both attempts failed
    if temp_c is None:

        # Display the clinic name and the reason for failure
        print(
            "Could not fetch temperature for:",
            clinic["clinic_name"],
            "-",
            error_reason
        )

    # Convert the original clinic row into a Python dictionary
    clinic_data = clinic.to_dict()

    # Add the fetched temperature to the clinic record
    # If unsuccessful, the value remains None
    clinic_data["temp_c"] = temp_c

    # Add the enriched clinic record to the results list
    enriched_rows.append(clinic_data)


# Convert all enriched clinic records into a pandas DataFrame
clinics_enriched = pd.DataFrame(enriched_rows)

# Display the enriched clinic dataset
display(clinics_enriched)

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,17.9
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.7
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.2
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,16.8
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,16.8
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,24.2
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,19.5
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,15.4


In [39]:
# Save the enriched clinic DataFrame as a CSV file
clinics_enriched.to_csv(
    "clinics_enriched.csv",
    index=False
)

# Confirm that the file has been saved
print("Saved clinics_enriched.csv")

Saved clinics_enriched.csv


In [40]:
# Display the number of rows and columns in the enriched dataset
print("Shape:", clinics_enriched.shape)

# Count the number of clinics for which temperature could not be fetched
print(
    "Missing temperatures:",
    clinics_enriched["temp_c"].isna().sum()
)

# Display the minimum and maximum available temperatures
print(
    "Temperature range:",
    clinics_enriched["temp_c"].min(),
    "to",
    clinics_enriched["temp_c"].max()
)

# Display the final column names
print("Columns:", clinics_enriched.columns.tolist())


Shape: (8, 7)
Missing temperatures: 0
Temperature range: 15.4 to 24.2
Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']


**Task 1.6 — Create final CSV and SQL files**

In [41]:
# Define the exact column order required by the assignment
final_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# Select only the required columns and create a separate copy
appointments_final = df[final_columns].copy()

# Sort the records by appointment_id
appointments_final = appointments_final.sort_values(
    by="appointment_id"
)

# Reset the row index after sorting
appointments_final = appointments_final.reset_index(drop=True)

# Convert appointment dates into YYYY-MM-DD format
appointments_final["booked_on"] = (
    appointments_final["booked_on"].dt.strftime("%Y-%m-%d")
)

appointments_final["appt_date"] = (
    appointments_final["appt_date"].dt.strftime("%Y-%m-%d")
)

# Display the final dataset
display(appointments_final.head())

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [42]:
# Check the number of rows and columns
print("Final shape:", appointments_final.shape)

# Check whether every appointment_id is unique
print(
    "Unique appointment IDs:",
    appointments_final["appointment_id"].is_unique
)

# Count missing values in every column
print("Missing values:")
print(appointments_final.isnull().sum())

# Save the final cleaned appointment data
appointments_final.to_csv(
    "appointments_clean.csv",
    index=False
)

# Confirm successful saving
print("Saved appointments_clean.csv")

Final shape: (990, 16)
Unique appointment IDs: True
Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
Saved appointments_clean.csv


In [43]:
# Read the enriched clinic CSV file
clinics_sql = pd.read_csv("clinics_enriched.csv")

# Display the clinic data before generating SQL
display(clinics_sql)

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,17.9
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.7
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.2
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,16.8
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,16.8
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,24.2
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,19.5
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,15.4


In [44]:
# Open the SQL file in write mode
# This creates a new file or replaces an existing file
with open("load_data.sql", "w", encoding="utf-8") as file:

    # Loop through each clinic record
    for _, row in clinics_sql.iterrows():

        # Convert missing temperature into SQL NULL
        if pd.isna(row["temp_c"]):
            temp = "NULL"
        else:
            # Format temperature to one decimal place
            temp = f"{float(row['temp_c']):.1f}"

        # Generate the SQL INSERT statement for this clinic
        sql = (
            f"INSERT INTO clinics VALUES "
            f"('{row['clinic_id']}', "
            f"'{row['clinic_name']}', "
            f"'{row['city']}', "
            f"'{row['tier']}', "
            f"{float(row['latitude']):.4f}, "
            f"{float(row['longitude']):.4f}, "
            f"{temp});\n"
        )

        # Write the SQL statement into the file
        file.write(sql)

print("Clinic INSERT statements written.")

Clinic INSERT statements written.


In [45]:
# Open the same SQL file in append mode
# Append mode adds statements without deleting clinic INSERT statements
with open("load_data.sql", "a", encoding="utf-8") as file:

    # Loop through all cleaned appointment records
    for _, row in appointments_final.iterrows():

        # Generate an INSERT statement for each appointment
        sql = (
            f"INSERT INTO appointments VALUES ("
            f"'{row['appointment_id']}', "
            f"'{row['patient_id']}', "
            f"'{row['clinic_id']}', "
            f"'{row['booked_on']}', "
            f"'{row['appt_date']}', "
            f"{int(row['age'])}, "
            f"'{row['gender']}', "
            f"{float(row['distance_km']):.1f}, "
            f"{int(row['fee_inr'])}, "
            f"'{row['booking_channel']}', "
            f"'{row['visit_type']}', "
            f"'{row['reminder_sent']}', "
            f"{int(row['prior_no_shows'])}, "
            f"'{row['status']}', "
            f"{int(row['lead_days'])}, "
            f"{int(row['no_show'])}"
            f");\n"
        )

        # Write the appointment INSERT statement
        file.write(sql)

print("Appointment INSERT statements written.")

Appointment INSERT statements written.


In [46]:
# Open the generated SQL file in read mode
with open("load_data.sql", "r", encoding="utf-8") as file:

    # Read all SQL statements
    sql_lines = file.readlines()

# Count all INSERT statements
insert_count = sum(
    1 for line in sql_lines
    if line.strip().upper().startswith("INSERT INTO")
)

# Display the total number of statements
print("Total INSERT statements:", insert_count)

# Display the first two clinic statements
print("\nFirst two SQL statements:")
print("".join(sql_lines[:2]))

# Display the last appointment statement
print("\nLast SQL statement:")
print(sql_lines[-1])

Total INSERT statements: 998

First two SQL statements:
INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 17.9);
INSERT INTO clinics VALUES ('C02', 'Hadapsar Care Centre', 'Pune', 'Metro', 18.5089, 73.9260, 18.7);


Last SQL statement:
INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 550, 'App', 'Follow-up', 'Yes', 2, 'No-Show', 45, 1);

